<a href="https://colab.research.google.com/github/cezmi9104-sys/Open-sourced-off-the-shelf-ion-exchange-membrane/blob/main/Gemma3_12b.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ============================================================
# HÜCRE 1 — Ollama Kurulum ve Gemma3 12B Yükleme
# ============================================================

# Önce zstd kur, sonra ollama
!sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

# Arka planda başlat
import subprocess, time
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
print("✅ Ollama çalışıyor!")

# Modeli indir
print("⏳ Gemma3 12B indiriliyor... (8GB, 5-10 dakika sürebilir)")
!ollama pull gemma3:12b
print("✅ Model hazır!")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 37 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (889 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 121852 files and directories currently i

In [3]:
# ============================================================
# HÜCRE 2 — Çeviri Fonksiyonları
# ============================================================
import requests

def translate_chunk(text: str, chunk_index: int = 0, total_chunks: int = 1) -> str:
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "gemma3:12b",
            "prompt": f"Aşağıdaki metni Türkçeye çevir. Sadece çeviriyi yaz, açıklama ekleme.\n\nMETİN:\n{text}\n\nTÜRKÇE ÇEVİRİ:",
            "stream": False,
        },
        timeout=300  # 5 dakika timeout
    )
    result = response.json()["response"].strip()
    print(f"  ✔ Parça {chunk_index + 1}/{total_chunks} çevrildi.")
    return result


def split_text(text: str, max_chars: int = 2500) -> list:
    paragraphs = [p.strip() for p in text.split("\n") if p.strip()]
    chunks, current = [], ""
    for para in paragraphs:
        if len(current) + len(para) + 1 <= max_chars:
            current += ("\n" if current else "") + para
        else:
            if current:
                chunks.append(current)
            if len(para) > max_chars:
                for i in range(0, len(para), max_chars):
                    chunks.append(para[i:i + max_chars])
            else:
                current = para
    if current:
        chunks.append(current)
    return chunks


# --- Test ---
print("🧪 Test çevirisi yapılıyor...")
test = translate_chunk("Hello, how are you?", 0, 1)
print(f"Sonuç: {test}")
print("\n📌 Artık Hücre 3'ü çalıştırabilirsiniz!")

🧪 Test çevirisi yapılıyor...
  ✔ Parça 1/1 çevrildi.
Sonuç: Merhaba, nasılsınız?

📌 Artık Hücre 3'ü çalıştırabilirsiniz!


In [7]:
# ============================================================
# HÜCRE 3 — PDF Çeviri
# ============================================================
!pip install -q pdfplumber

import pdfplumber
from google.colab import files

# PDF yükle
print("📂 PDF dosyası seçin:")
uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]

# PDF'den metin çıkar
full_text = ""
with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        text = page.extract_text()
        if text:
            full_text += text + "\n"

print(f"📄 {len(full_text)} karakter okundu.")

# Parçalara böl ve çevir
chunks = split_text(full_text)
print(f"✂️  {len(chunks)} parçaya bölündü. Çeviri başlıyor...\n")

translated_parts = []
for i, chunk in enumerate(chunks):
    translated = translate_chunk(chunk, i, len(chunks))
    translated_parts.append(translated)

final_translation = "\n\n".join(translated_parts)

# Sonucu kaydet
output_file = pdf_path.replace(".pdf", "_turkce.txt")
with open(output_file, "w", encoding="utf-8") as f:
    f.write(final_translation)

print(f"\n✅ Çeviri tamamlandı!")
print(f"📥 İndiriliyor: {output_file}")
files.download(output_file)

📂 PDF dosyası seçin:


Saving CN102299326B.pdf to CN102299326B.pdf
📄 0 karakter okundu.
✂️  0 parçaya bölündü. Çeviri başlıyor...


✅ Çeviri tamamlandı!
📥 İndiriliyor: CN102299326B_turkce.txt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>